In [1]:
import nest_asyncio
nest_asyncio.apply()

## Config

In [2]:
from pydantic import BaseModel
from agents import (
    AsyncOpenAI,
    OpenAIChatCompletionsModel,
    RunConfig
)
#from google.colab import userdata


In [3]:
#gemini_api_key = userdata.get("GEMINI_API_KEY")
import os
gemini_api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")


# Check if the API key is present; if not, raise an error
if not gemini_api_key:
    raise ValueError("GEMINI_API_KEY is not set. Please ensure it is defined in your .env file.")

#Reference: https://ai.google.dev/gemini-api/docs/openai
external_client = AsyncOpenAI(
    api_key=gemini_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

model = OpenAIChatCompletionsModel(
    model="gemini-2.0-flash",
    openai_client=external_client
)

AGENT_MODEL=model
config = RunConfig(
    model=model,
    model_provider=external_client,
    tracing_disabled=True
)

# Creating a handoff

All agents have a handoffs param, which can either take an Agent directly, or a Handoff object that customizes the Handoff.

In [4]:
import asyncio
from agents import Agent, handoff, Runner

In [5]:
from crewai_tools import ScrapeWebsiteTool
from agents import Agent, FunctionTool, RunContextWrapper, function_tool

@function_tool
def scrape_website1(website_url: str) -> str:
    """
    Scrapes the main textual content from a given website URL.
    """
    tool = ScrapeWebsiteTool(website_url=website_url)
    return tool.run()


/home/tjamil/Insync/MyLearning/Dev202xAgents/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
#scrape_website("https://aibytec.com/")

In [7]:
@function_tool
async def search_tool(query: str) -> str:
    """Useful to search the internet about a given topic and return relevant results."""
    print(f"--- Using Tool: search_internet with query: '{query}' ---")
    try:
        url = "https://google.serper.dev/search"
        payload = json.dumps({"q": query})
        headers = {'X-API-KEY': os.environ['SERPER_API_KEY'], 'Content-Type': 'application/json'}
        
        async with httpx.AsyncClient(timeout=30.0) as client:
            response = await client.post(url, headers=headers, content=payload)
        response.raise_for_status()
        response_json = response.json()
        
        if 'organic' not in response_json or not response_json['organic']: 
            return "No organic results found."
            
        results = response_json['organic'][:4]
        string = [f"Title: {r.get('title', 'N/A')}\nLink: {r.get('link', 'N/A')}\nSnippet: {r.get('snippet', 'N/A')}\n---" for r in results]
        return '\n'.join(string)
    except Exception as e:
        print(f"Search error: {e}")
        return f"An error occurred during search: {e}"


In [8]:
import asyncio
from crewai_tools import ScrapeWebsiteTool

@function_tool
async def scrape_tool(website_url: str) -> str:
    """Scrapes the content of a given website URL."""
    print(f"--- Using Tool: scrape_website with URL: '{website_url}' ---")
    try:
        scraper_instance = ScrapeWebsiteTool()
        loop = asyncio.get_running_loop()
        
        result = await asyncio.wait_for(
            loop.run_in_executor(
                None, 
                lambda: scraper_instance._run(website_url=website_url)
            ),
            timeout=60.0
        )
        return str(result)[:5000]  # Truncate to 5000 characters
    except asyncio.TimeoutError:
        return f"Scraping timed out for {website_url}"
    except Exception as e:
        print(f"Scraping error: {e}")
        return f"An error occurred during scraping: {e}"


### 1. Basic Usage

In [9]:
from agents import Agent, handoff
from pydantic import BaseModel, Field

# -----------------------------
# Pydantic Model for Final Output
# -----------------------------
class Copy(BaseModel):
    """Copy model"""
    title: str = Field(description="Title of the copy")
    body: str = Field(description="Body of the copy")

# -----------------------------
# Define Agents with Deterministic Handoff
# -----------------------------

chief_creative_director = Agent(
    name="chief_creative_director",
    model=AGENT_MODEL,
    instructions=(
        "As the Chief Creative Director, your role is to synthesize the entire marketing workflow into a polished report. "
        "You will receive the market research, marketing strategy, and campaign ideas. "
        "Carefully review each component, identify gaps or inconsistencies, and integrate them into a cohesive marketing report. "
        "Ensure the final output flows logically, reads professionally, and aligns with business goals. "
        "Return your final output as a JSON object with two fields: `title` for the report title, and `body` for the full content in markdown format."
    ),
    output_type=Copy
)

creative_content_creator = Agent(
    name="creative_content_creator",
    model=AGENT_MODEL,
    instructions=(
        "As the Creative Content Creator, your task is to transform the marketing strategy into exactly five original marketing campaign ideas. "
        "Each campaign should have a clear and catchy title, a concise but creative description, and a statement of the expected impact (e.g., engagement, brand awareness, or conversion). "
        "Write in markdown format, using headings for each idea. "
        "After completing the campaigns, hand off your output to the Chief Creative Director for final synthesis."
    ),
    handoffs=[chief_creative_director]
)

chief_marketing_strategist = Agent(
    name="chief_marketing_strategist",
    model=AGENT_MODEL,
    instructions=(
        "As the Chief Marketing Strategist, your job is to interpret the market research and develop a comprehensive marketing strategy. "
        "Your strategy should include target audience definition, positioning, messaging pillars, and SMART objectives. "
        "Avoid general placeholders — instead, make each section concrete, relevant, and backed by insights. "
        "Use markdown format and structure your strategy clearly. "
        "Once complete, hand off the strategy to the Creative Content Creator."
    ),
    tools=[search_tool],
    handoffs=[creative_content_creator]
)

lead_market_analyst = Agent(
    name="lead_market_analyst",
    model=AGENT_MODEL,
    instructions=(
        "As the Lead Market Analyst, you are responsible for conducting a detailed analysis of the market landscape. "
        "Use the `search_internet` tool to collect current, relevant information on the client’s industry, competitors, and market trends. "
        "Use the `scrape_website` tool to gather detailed insights from official company and competitor websites. "
        "Ensure your analysis includes strengths, weaknesses, opportunities, and threats (SWOT), and conclude with key actionable insights. "
        "Format your report in markdown. After completing the analysis, hand it off to the Chief Marketing Strategist."
    ),
    tools=[search_tool, scrape_tool],
    handoffs=[chief_marketing_strategist]
)


In [10]:
async def main(input: str):
    result = await Runner.run(chief_creative_director, input=input, run_config=config)
    print(result.final_output)
    text = f"""{result.final_output.title}\n
    {result.final_output.body}\n"""
    with open("Report.md", "w") as f:
        f.write(text)


In [11]:
current_context = (
        "Customer Domain: https://aibytec.com/\n\n"
        "Project Description: Develop a marketing strategy for AiByTec's advanced AI-driven solutions, "
        "targeting tech-savvy decision-makers specially in local pakistan markt."
)

asyncio.run(main("current_context"))

title='Integrated Marketing Report: Project Phoenix' body='# Integrated Marketing Report: Project Phoenix\n\n## Executive Summary\n\nThis report synthesizes market research, marketing strategy, and campaign ideas for "Project Phoenix," outlining a comprehensive approach to [Specify Business Goal, e.g., increasing market share, launching a new product, improving brand awareness]. It identifies key target audiences, proposes strategic marketing initiatives, and details specific campaigns designed to achieve measurable results. This report aims to ensure alignment across all marketing activities, maximizing ROI and driving sustainable growth.\n\n## 1. Market Research Summary\n\n*   **Target Audience:** [Detailed description of target audience demographics, psychographics, needs, and pain points. E.g., Millennials aged 25-35, tech-savvy, interested in sustainable living, seeking affordable and convenient solutions].\n*   **Market Trends:** [Analysis of current market trends relevant to the